# KALIA v0.3.0 Phase A micros — X17 doc-mask + code-10 mixture

Two cheap mechanism checks that gate Phase B: X17 (registered) decides whether document masking enters the v0.2.1 pipeline; code-10 decides whether the code share rises 5% -> 10%.

Both run at micro scale against their registered bars. Neither touches the Step 1-2 kernels.

In [ ]:
import glob, os, shutil, subprocess, sys
work = "/kaggle/working/kalia"
if os.path.exists(work):
    shutil.rmtree(work)
r = subprocess.run(["git", "clone", "--quiet", "--branch", "main", "https://github.com/subhajitlucky/kalia.git", work], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
os.chdir(work)
print("code ready")
data = sorted(glob.glob("/kaggle/input/**/train.bin", recursive=True))
assert data, "train.bin not found - attach kalia-v030-inputs"
DATA = os.path.dirname(data[0])
print("data:", DATA)
import torch
print("gpu:", torch.cuda.is_available())
code10 = sorted(glob.glob("/kaggle/input/**/code10.bin", recursive=True))
CODE10 = os.path.dirname(code10[0]) if code10 else None
print("code10 split:", CODE10 if CODE10 else "MISSING - X17 only this run")


In [ ]:
print(subprocess.run([sys.executable, "ablate.py", "--arms", "micro-base,micro-docmask", "--data-dir", DATA, "--out-root", "/kaggle/working/out"], capture_output=True, text=True).stdout[-4000:])


In [ ]:
import pandas as pd
for a in ["micro-base", "micro-docmask"]:
    df = pd.read_csv(f"/kaggle/working/out/{a}/val_log.csv")
    print(f"{a:16s} final={float(df.iloc[-1].val_loss):.4f}")
print("\nX17 bar: docmask beats control by >= 0.010 nats")


In [ ]:
if not CODE10:
    print("code10.bin not attached - X17 only this run; publish the split for the code-10 pair")
else:
    print(subprocess.run([sys.executable, "ablate.py", "--arms", "micro-base,micro-code10", "--data-dir", CODE10, "--out-root", "/kaggle/working/out-code10"], capture_output=True, text=True).stdout[-4000:])
